# War Crimes & Human Rights Violations — EDA

**This notebook is the primary interface for the project.** Run cells top-to-bottom.
No terminal commands required — data ingestion, all analysis, and all chart exports
run from here.

**Sources:** ACLED (Armed Conflict Location & Event Data) + HRDAG (Human Rights Data Analysis Group)  
**Event types:** Battles | Explosions/Remote violence | Violence against civilians | Sexual violence  

**Research questions**
1. Who are the actors (state vs. non-state) committing the most documented violations?
2. What event types are most common, and has that changed over time?
3. When and where are violations most concentrated geographically?

---

> **One-time setup required before running:**
> 1. Copy `.env.example` → `.env` in the project root
> 2. Add your `ACLED_EMAIL` and `ACLED_API_KEY` (free at [acleddata.com/register](https://acleddata.com/register/))
> 3. Run all cells in order

## 0. Install & Import

In [ ]:
# Install dependencies if not already present
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(__import__('pathlib').Path('..').resolve() / 'requirements.txt')
])
print('Dependencies ready.')

In [ ]:
import warnings, sys
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # save files without a display
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
import folium
from folium.plugins import HeatMap, MarkerCluster
from IPython.display import display, IFrame, HTML

# Add project root so src/ modules are importable
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.ingest import ingest_acled, ingest_hrdag_colombia, ingest_hrdag_guatemala
from src.visualize import (
    load_acled, ICC_SITUATION_COUNTRIES, _classify_actor_type, PALETTE, OUT_DIR, PROC_DIR,
    chart_choropleth_world, chart_event_cluster_map, chart_heatmap_density,
    chart_monthly_events_by_type, chart_animated_timeseries, chart_yoy_violence_civilians,
    chart_top20_actors, chart_actor_type_by_region, chart_actor_network,
    chart_accountability_gap, chart_data_completeness, export_summary,
)

RAW_DIR  = ROOT / 'data' / 'raw'
sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 50)
print('Imports complete.')

---
## 1. Data Ingestion

Pulls ACLED (last 5 years, global) and HRDAG (Colombia + Guatemala) into `data/raw/`.
Skip this section if the files already exist and you just want to re-run analysis.

In [ ]:
# ── ACLED ──────────────────────────────────────────────────────────────────
# Requires ACLED_EMAIL + ACLED_API_KEY in ../.env
# Skips download if file already exists (delete to force refresh)
acled_path = RAW_DIR / 'acled_raw.csv'

if acled_path.exists():
    print(f'ACLED file already present ({acled_path.stat().st_size / 1e6:.1f} MB). '
          f'Delete it to force a fresh pull.')
else:
    print('Pulling ACLED data — this may take a few minutes...')
    df_raw = ingest_acled()
    print(f'Done: {len(df_raw):,} rows')

In [ ]:
# ── HRDAG ──────────────────────────────────────────────────────────────────
# Public datasets — no API key needed.
# If URLs change, see fallback instructions printed by the function.
for path, fn, label in [
    (RAW_DIR / 'hrdag_colombia.csv',  ingest_hrdag_colombia,  'Colombia'),
    (RAW_DIR / 'hrdag_guatemala.csv', ingest_hrdag_guatemala, 'Guatemala'),
]:
    if path.exists():
        print(f'HRDAG {label} already present — skipping download.')
    else:
        print(f'Downloading HRDAG {label}...')
        fn()

---
## 2. Load & Inspect

In [ ]:
df = load_acled(RAW_DIR / 'acled_raw.csv')

if df.empty:
    raise RuntimeError(
        'No ACLED data found. '
        'Check that your .env credentials are set and re-run Section 1.'
    )

print(f'ACLED: {len(df):,} rows | '
      f'{df["country"].nunique()} countries | '
      f'{df["event_date"].min().date()} → {df["event_date"].max().date()}')
display(df.head(3))

In [ ]:
# HRDAG
df_col  = pd.read_csv(RAW_DIR / 'hrdag_colombia.csv')  if (RAW_DIR / 'hrdag_colombia.csv').exists()  else pd.DataFrame()
df_guat = pd.read_csv(RAW_DIR / 'hrdag_guatemala.csv') if (RAW_DIR / 'hrdag_guatemala.csv').exists() else pd.DataFrame()

print(f'HRDAG Colombia:  {len(df_col):,} rows  | cols: {list(df_col.columns[:6])  if not df_col.empty  else "(not loaded)"}')
print(f'HRDAG Guatemala: {len(df_guat):,} rows | cols: {list(df_guat.columns[:6]) if not df_guat.empty else "(not loaded)"}')

In [ ]:
# Schema + missing-value audit
missing = df.isnull().sum().sort_values(ascending=False)
display(
    pd.DataFrame({'missing': missing, 'pct': (missing / len(df) * 100).round(2)})
    .query('missing > 0')
)

print('\nEvent type distribution:')
print(df['event_type'].value_counts().to_string())

---
## 3. Research Question 1 — Who Are the Actors?

ACLED `inter1` codes: 1=State Forces, 2=Rebel Groups, 3=Political Militias,
4=Communal Militias, 5=Rioters, 6=Protesters, 7=Civilians, 8=External/Other

In [ ]:
if 'inter1' in df.columns:
    df['actor_type'] = df['inter1'].apply(_classify_actor_type)
    df['state_vs_nonstate'] = df['actor_type'].apply(
        lambda x: 'State' if x in ('State Military', 'State Police') else 'Non-State'
    )

actor_counts = df['actor_type'].value_counts().reset_index()
actor_counts.columns = ['Actor Type', 'Events']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(actor_counts['Actor Type'], actor_counts['Events'], color='#c0392b', alpha=0.85)
axes[0].set_title('Events by Actor Type (Primary Actor)', fontsize=13)
axes[0].set_xlabel('Event Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

axes[1].pie(
    actor_counts['Events'],
    labels=actor_counts['Actor Type'],
    autopct='%1.1f%%',
    colors=sns.color_palette('Set2', len(actor_counts)),
    startangle=90,
)
axes[1].set_title('Share of Events by Actor Type', fontsize=13)

plt.tight_layout()
plt.savefig(OUT_DIR / 'actor_type_overview.png', dpi=150)
plt.show()
display(actor_counts)

In [ ]:
# Top 20 named actors
top20 = df['actor1'].value_counts().head(20).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top20.index, top20.values, color='#c0392b', alpha=0.85)
ax.set_title('Top 20 Named Actors by Event Count (ACLED)', fontsize=13)
ax.set_xlabel('Events')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, top20.values):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'top20_actors.png', dpi=150)
plt.show()

In [ ]:
# State vs. non-state by event type
if 'state_vs_nonstate' in df.columns:
    pivot = (
        df.groupby(['event_type', 'state_vs_nonstate'])
        .size()
        .unstack(fill_value=0)
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot.plot(kind='bar', ax=ax, color=['#e74c3c', '#3498db'], edgecolor='white')
    ax.set_title('State vs. Non-State Actors by Event Type', fontsize=13)
    ax.set_xlabel('')
    ax.set_ylabel('Events')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='Actor Category')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'state_vs_nonstate.png', dpi=150)
    plt.show()
    display(pivot)

In [ ]:
# Actor type stacked bar by region — saved to outputs/actor_type_by_region.html
chart_actor_type_by_region(df)
display(HTML('<a href="../outputs/actor_type_by_region.html" target="_blank">Open actor_type_by_region.html</a>'))

### Actor Network (Force-Directed)

Layout uses the **Fruchterman-Reingold** spring algorithm (NetworkX `spring_layout`).
Edge weight = co-occurrence frequency; heavily-interacting pairs cluster together.
Node size = total event count. Color = actor type.

In [ ]:
chart_actor_network(df, top_n=40)
display(HTML('<a href="../outputs/actor_network.html" target="_blank">Open actor_network.html</a>'))

---
## 4. Research Question 2 — Event Types Over Time

In [ ]:
df['month'] = df['event_date'].dt.to_period('M').dt.to_timestamp()

monthly = (
    df.groupby(['month', 'event_type'])
    .size()
    .reset_index(name='count')
)

fig, ax = plt.subplots(figsize=(14, 5))
for etype, color in PALETTE.items():
    sub = monthly[monthly['event_type'] == etype]
    ax.plot(sub['month'], sub['count'], label=etype, color=color, linewidth=2)
ax.set_title('Monthly Events by Type (ACLED)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Events')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(title='Event Type', loc='upper left')
plt.tight_layout()
plt.savefig(OUT_DIR / 'monthly_events_static.png', dpi=150)
plt.show()

In [ ]:
# Interactive monthly line chart — saved to outputs/
chart_monthly_events_by_type(df)
display(HTML('<a href="../outputs/monthly_events_by_type.html" target="_blank">Open monthly_events_by_type.html</a>'))

In [ ]:
# Annual totals and YoY change
annual = (
    df.groupby(['year', 'event_type'])
    .size()
    .unstack(fill_value=0)
)
print('Annual event counts by type:')
display(annual)
print('\nYear-over-year % change:')
display(annual.pct_change().mul(100).round(1))

In [ ]:
# YoY bar chart for violence against civilians
chart_yoy_violence_civilians(df)
display(HTML('<img src="../outputs/yoy_violence_civilians.png" width="800">'))

In [ ]:
# Sexual violence trend with 2020 methodology note
sv = df[df['event_type'] == 'Sexual violence']
if not sv.empty:
    sv_annual = sv.groupby('year').size()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(sv_annual.index.astype(int), sv_annual.values, color='#2471a3', alpha=0.85)
    ax.axvline(x=2020, color='red', linestyle='--', linewidth=1.2, alpha=0.8,
               label='ACLED sexual violence category introduced (2020)')
    ax.set_title('Annual Sexual Violence Events (ACLED)\n'
                 'Pre-2020 data is structurally incomplete — category not yet tracked', fontsize=12)
    ax.set_xlabel('Year')
    ax.set_ylabel('Events')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'sexual_violence_trend.png', dpi=150)
    plt.show()
else:
    print('No sexual violence events in dataset.')

In [ ]:
# Animated choropleth: hotspot shift year over year
chart_animated_timeseries(df)
display(HTML('<a href="../outputs/animated_timeseries.html" target="_blank">Open animated_timeseries.html</a>'))

---
## 5. Research Question 3 — Geographic Concentration

In [ ]:
# Top 20 countries
top_countries = df['country'].value_counts().head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_countries.index[::-1], top_countries.values[::-1], color='#e67e22', alpha=0.85)
ax.set_title('Top 20 Countries by Event Count (ACLED)', fontsize=13)
ax.set_xlabel('Events')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(OUT_DIR / 'top20_countries.png', dpi=150)
plt.show()

In [ ]:
# Choropleth world map
chart_choropleth_world(df)
display(HTML('<a href="../outputs/choropleth_world.html" target="_blank">Open choropleth_world.html</a>'))

In [ ]:
# Event density heatmap
chart_heatmap_density(df)
display(HTML('<a href="../outputs/heatmap_density.html" target="_blank">Open heatmap_density.html</a>'))

In [ ]:
# Dot/cluster map with tooltips
chart_event_cluster_map(df)
display(HTML('<a href="../outputs/event_cluster_map.html" target="_blank">Open event_cluster_map.html</a>'))

In [ ]:
# Country x event type matrix heatmap (top 15 countries)
top15 = df['country'].value_counts().head(15).index
heat_data = (
    df[df['country'].isin(top15)]
    .groupby(['country', 'event_type'])
    .size()
    .unstack(fill_value=0)
)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(heat_data, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Events'})
ax.set_title('Event Type × Country Matrix (Top 15 Countries)', fontsize=13)
ax.set_xlabel('Event Type')
ax.set_ylabel('Country')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(OUT_DIR / 'country_event_matrix.png', dpi=150)
plt.show()

In [ ]:
# Regional trend lines
if 'region' in df.columns:
    region_year = (
        df.groupby(['year', 'region'])
        .size()
        .reset_index(name='events')
    )
    fig = px.line(
        region_year, x='year', y='events', color='region',
        title='Annual Events by Region (ACLED)',
        labels={'events': 'Events', 'year': 'Year'},
        template='plotly_white',
    )
    fig.write_html(str(OUT_DIR / 'regional_trends.html'))
    fig.show()
    display(HTML('<a href="../outputs/regional_trends.html" target="_blank">Open regional_trends.html</a>'))

---
## 6. Accountability Gap

Which countries have high documented event counts but **no open ICC situation**?
ICC situation list current as of 2024. See README for structural reasons behind the gap.

In [ ]:
country_counts = df['country'].value_counts().head(30).reset_index()
country_counts.columns = ['country', 'events']
country_counts['icc_situation'] = country_counts['country'].apply(
    lambda c: 'ICC Open Situation' if c in ICC_SITUATION_COUNTRIES else 'No ICC Situation'
)

no_icc = country_counts[country_counts['icc_situation'] == 'No ICC Situation']
has_icc = country_counts[country_counts['icc_situation'] == 'ICC Open Situation']

print(f'Top 30 countries — with ICC situation: {len(has_icc)} | without: {len(no_icc)}')
print('\nHigh-event countries with NO ICC situation:')
display(no_icc.head(10))

In [ ]:
chart_accountability_gap(df)
display(HTML('<a href="../outputs/accountability_gap.html" target="_blank">Open accountability_gap.html</a>'))

---
## 7. Data Completeness

Composite proxy scores combining:
- **RSF World Press Freedom Index (2023)** — low press freedom = fewer events captured (Weidmann, 2016)
- **ACLED source-coverage tier** — regions with denser NGO/monitor networks score higher (ACLED Codebook, 2024)
- **Inter-source divergence** — Eck (2012) compared ACLED vs. UCDP GED; high divergence signals undercount uncertainty
- **Sexual violence column** additionally penalised for stigma-based underreporting (Cohen & Green, 2012; Fariss, 2014)

These are **ordinal proxies**, not official measurements.

In [ ]:
chart_data_completeness(df)
display(HTML('<img src="../outputs/data_completeness.png" width="850">'))

---
## 8. HRDAG Deep Dive

HRDAG uses **multiple systems estimation (MSE)** to estimate total killings
including undocumented cases — a fundamentally different methodology from ACLED.
These cells adapt to whatever columns the downloaded dataset contains.

In [ ]:
for label, df_h in [('Colombia', df_col), ('Guatemala', df_guat)]:
    if df_h.empty:
        print(f'HRDAG {label}: not loaded — re-run Section 1 ingestion.')
        continue

    print(f'\n=== HRDAG {label} ===')
    print(f'{len(df_h):,} rows | {df_h.shape[1]} columns')
    display(df_h.head(3))

    # Perpetrator column (common HRDAG naming conventions)
    perp_col = next(
        (c for c in df_h.columns if any(k in c.lower() for k in ('perp', 'actor', 'grupo', 'armed'))),
        None
    )
    if perp_col:
        pc = df_h[perp_col].value_counts().head(12)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.barh(pc.index, pc.values, color='#8e44ad', alpha=0.85)
        ax.set_title(f'HRDAG {label}: Records by Perpetrator ({perp_col})', fontsize=12)
        ax.set_xlabel('Records')
        plt.tight_layout()
        plt.savefig(OUT_DIR / f'hrdag_{label.lower()}_perpetrators.png', dpi=150)
        plt.show()

    # Year column
    year_col = next(
        (c for c in df_h.columns if any(k in c.lower() for k in ('year', 'año', 'date', 'fecha'))),
        None
    )
    if year_col:
        try:
            df_h['_year'] = pd.to_numeric(df_h[year_col], errors='coerce')
            annual = df_h.dropna(subset=['_year']).groupby('_year').size()
            fig, ax = plt.subplots(figsize=(12, 3))
            ax.plot(annual.index, annual.values, marker='o', color='#8e44ad', linewidth=2)
            ax.set_title(f'HRDAG {label}: Annual Documented Records', fontsize=12)
            ax.set_xlabel('Year')
            ax.set_ylabel('Records')
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
            plt.tight_layout()
            plt.savefig(OUT_DIR / f'hrdag_{label.lower()}_annual.png', dpi=150)
            plt.show()
        except Exception as e:
            print(f'  Could not plot year trend: {e}')

---
## 9. Summary & Export

In [ ]:
print('=== Global ACLED Summary ===')
print(f'Total events:        {len(df):,}')
print(f'Total countries:     {df["country"].nunique()}')
print(f'Total fatalities:    {df["fatalities"].sum():,}')
print(f'Date range:          {df["event_date"].min().date()} → {df["event_date"].max().date()}')

print('\n=== Top 10 High-Event Countries With No ICC Situation ===')
display(
    df.groupby('country').size().reset_index(name='events')
    .sort_values('events', ascending=False)
    .assign(icc=lambda x: x['country'].isin(ICC_SITUATION_COUNTRIES))
    .query('icc == False')
    .head(10)
    .reset_index(drop=True)
)

In [ ]:
# Export all summary CSVs
export_summary(df)
print('Exported to data/processed/ and outputs/summary_aggregated.csv')

---
## 10. Findings

*(Update these after running with your actual data.)*

### Actors
- Non-state armed groups (rebel groups, militias) account for the plurality of documented events.
- State military actors appear disproportionately in [region] vs. others.
- Top named actors by event count: see `outputs/top20_actors.png`.

### Event Types Over Time
- Battles consistently represent the largest share of events.
- Violence against civilians has [trended up/down] since [year].
- Sexual violence shows a sharp increase after 2020 — a reporting artifact, not
  a real-world change (ACLED methodology change).

### Geography
- Highest event concentrations: [countries].
- [Region] saw the largest year-over-year increase.

### Accountability Gap
- Several high-event countries have no open ICC situation, including [list].
- This reflects ICC jurisdiction limits, Security Council veto dynamics, and
  non-ratification of the Rome Statute.

### Methodological Caveats
1. ACLED counts **events**, not victims. Event count ≠ severity.
2. Sexual violence category introduced 2020; pre-2020 comparisons unreliable.
3. Low-press-freedom regions are systematically undercounted (Weidmann, 2016).
4. HRDAG covers historical conflicts only; not directly comparable to ACLED real-time data.
5. ICC situation list current as of 2024.